In [ ]:
https://medium.com/@cole.ian.diamond/grokking-two-tower-models-53e0140897e2

In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "pyproject.toml").is_file()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.load_dataset import laod_dataframes

from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from surprise import SVD, Dataset, Reader

In [7]:
dataset_dict = laod_dataframes()

books = dataset_dict["df_books"]
ratings = dataset_dict["df_ratings"]
df_book_tags = dataset_dict["df_book_tags"]
df_tags = dataset_dict["df_tags"]
df_to_read = dataset_dict["df_to_read"]

In [6]:
# -----------------------------
# 1) Filter users by activity
# -----------------------------
# Count ratings per user
user_counts = ratings["user_id"].value_counts()

# Keep users with between 10 and 1000 ratings for a denser, more stable toy dataset
active_users = user_counts[(user_counts >= 10) & (user_counts <= 1000)].index

# Filter the ratings table down to those users
mini = ratings[ratings["user_id"].isin(active_users)].copy()

# -----------------------------
# 2) Filter books by popularity
# -----------------------------
# Count ratings per book within the filtered user subset
book_counts = mini["book_id"].value_counts()

# Keep books with at least 50 interactions in this subset
popular_books = book_counts[book_counts >= 50].index

# Filter again
mini = mini[mini["book_id"].isin(popular_books)].copy()

# -----------------------------
# 3) Label: Convert rating → implicit like if rating >= 4
# -----------------------------
mini["label"] = (mini["rating"] >= 4).astype("int8")

In [8]:
# -----------------------------
# 4) Keep only book columns we want + feature engineering
# -----------------------------
book_feats = books[["book_id", "original_publication_year", "language_code", "authors"]].copy()

# Keep a small set of metadata columns we want to use as item features
book_feats = books[["book_id", "original_publication_year", "language_code", "authors"]].copy()

# Ensure publication year is numeric; non-numeric becomes NaN
book_feats["original_publication_year"] = pd.to_numeric(
    book_feats["original_publication_year"], errors="coerce"
)

# Bucket year by decade: e.g., 1999 -> 1990, 2003 -> 2000
book_feats["year_bucket"] = (
    (book_feats["original_publication_year"] // 10) * 10
)

# Extract a single "primary author" by taking the first in the comma-separated list
book_feats["primary_author"] = (
    book_feats["authors"]
    .astype(str)
    .str.split(",")
    .str[0]
    .str.strip()
)

# -----------------------------
# 5) Merge book metadata (year + language + author)
# -----------------------------
mini = mini.merge(book_feats, on="book_id", how="left")

# -----------------------------
# 6) Fill missing language/year AFTER merge
# -----------------------------

mini["language_code"] = (
    mini["language_code"]
    .fillna("UNK")
    .astype(str)
    .str.strip()
)
mini.loc[mini["language_code"] == "", "language_code"] = "UNK"

# Year bucket: add explicit UNK bucket for missing (fine to fill here)
mini["year_bucket"] = mini["year_bucket"].fillna(-1).astype(np.int32)

# build author vocabulary
mini["primary_author"] = (
    mini["primary_author"]
    .fillna("UNK")
    .astype(str)
)

# -----------------------------
# 7) Encodings
# -----------------------------
# e.g. encode language to integer ids (0..num_langs-1)
mini["lang_idx"] = mini["language_code"].astype("category").cat.codes.astype(np.int64)
mini["user_idx"] = mini["user_id"].astype("category").cat.codes.astype(np.int64)
mini["item_idx"] = mini["book_id"].astype("category").cat.codes.astype(np.int64)
mini["year_idx"] = mini["year_bucket"].astype("category").cat.codes.astype(np.int64)

author_to_idx = {a: i for i, a in enumerate(mini["primary_author"].unique())}
mini["author_idx"] = mini["primary_author"].map(author_to_idx).astype(np.int64)

display(mini.head())

,book_id,user_id,rating,label,original_publication_year,language_code,authors,year_bucket,primary_author,lang_idx,user_idx,item_idx,year_idx,author_idx
0,1,314,5,1,2005.0,eng,"J.K. Rowling, Mary GrandPré",2000,J.K. Rowling,4,113,0,35,0
1,1,439,3,0,2005.0,eng,"J.K. Rowling, Mary GrandPré",2000,J.K. Rowling,4,181,0,35,0
2,1,588,5,1,2005.0,eng,"J.K. Rowling, Mary GrandPré",2000,J.K. Rowling,4,246,0,35,0
3,1,1169,4,1,2005.0,eng,"J.K. Rowling, Mary GrandPré",2000,J.K. Rowling,4,510,0,35,0
4,1,1185,4,1,2005.0,eng,"J.K. Rowling, Mary GrandPré",2000,J.K. Rowling,4,518,0,35,0


In [9]:
# -----------------------------
# 8) Display stats
# -----------------------------

num_users = mini["user_idx"].nunique()
num_items = mini["item_idx"].nunique()
num_langs = mini["lang_idx"].nunique()
num_authors = mini["author_idx"].nunique()
num_year_buckets = mini["year_idx"].nunique()
num_interactions = len(mini)

positives = (mini["label"] == 1).sum()
positive_rate = (mini["label"] == 1).mean()

unique_user_ids = mini["user_id"].nunique()
unique_book_ids = mini["book_id"].nunique()

# --- build stats dict + DataFrame ---
stats = {
    "num_users": num_users,
    "num_items": num_items,
    "num_langs": num_langs,
    "num_authors": num_authors,
    "num_year_buckets": num_year_buckets,
    "num_interactions": num_interactions,
    "positives": positives,
    "positive_rate": positive_rate,
    "unique_user_ids": unique_user_ids,
    "unique_book_ids": unique_book_ids,
}

stats_df = pd.DataFrame(stats, index=["value"]).T.astype(str)

# --- pretty formatting ---
for col in set(stats.keys()).difference({"positive_rate"}):
    stats_df.loc[col, "value"] = f"{stats[col]:,}"
stats_df.loc["positive_rate", "value"] = f"{positive_rate:.3f}"

display(stats_df)

,value
num_users,"24,361"
num_items,"9,145"
num_langs,7
num_authors,378
num_year_buckets,36
num_interactions,"827,028"
positives,"545,668"
positive_rate,0.660
unique_user_ids,"24,361"
unique_book_ids,"9,145"


In [10]:
from sklearn.model_selection import train_test_split

seed = 42

# Create training set (80% of data)
train_df, temp_df = train_test_split(
    mini,
    test_size=0.2,
    random_state=seed,
    stratify=mini["label"],
)

# Create validation and testing set from equal halves of the remaining 20% of data
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=seed,
    stratify=temp_df["label"],
)

print(len(train_df), len(val_df), len(test_df))

display(train_df.head(5))

661622 82703 82703


,book_id,user_id,rating,label,original_publication_year,language_code,authors,year_bucket,primary_author,lang_idx,user_idx,item_idx,year_idx,author_idx
311869,3160,47468,5,1,NaN,UNK,NaN,-1,UNK,0,21950,3152,9,1
3857,39,30601,5,1,NaN,UNK,NaN,-1,UNK,0,14681,38,9,1
145601,1463,10915,5,1,NaN,UNK,NaN,-1,UNK,0,5324,1461,9,1
423958,4361,3721,4,1,NaN,UNK,NaN,-1,UNK,0,1778,4328,9,1
780423,9047,19005,3,0,NaN,UNK,NaN,-1,UNK,0,9196,8497,9,1


In [11]:
import torch
from torch.utils.data import Dataset, DataLoader
import random

class GoodreadsTwoTowerDataset(Dataset):
    """
    Yields (user, positive_item, negative_items) for two-tower training.

    We expect df_pos to contain ONLY positive interactions.
    Negative items are sampled uniformly from all items, excluding items the user liked.
    """
    def __init__(
        self,
        df_pos,
        user_to_pos_set,
        num_items,
        num_neg=5,
        seed=42,
    ):
        # Store arrays of users and positive items for indexing
        self.users = df_pos["user_idx"].values
        self.pos_items = df_pos["item_idx"].values

        # Dict: user -> set of positive items (to avoid sampling positives as negatives)
        self.user_to_pos = user_to_pos_set

        # Total item universe size
        self.num_items = int(num_items)

        # How many negatives to sample per positive
        self.num_neg = int(num_neg)

        # RNG used for negative sampling (deterministic given seed)
        self.rng = random.Random(seed)

    def __len__(self):
        # Number of positive examples
        return len(self.users)

    def __getitem__(self, idx):
        # Grab one positive training row
        u = int(self.users[idx])
        i_pos = int(self.pos_items[idx])

        # User’s known positives (to avoid)
        pos_set = self.user_to_pos.get(u, set())

        # Sample negatives
        negs = []
        tries = 0

        # In rare cases, a user might have liked a huge fraction of items
        # making it hard to find unseen items. 
        # The tries limit prevents an infinite loop.
        while len(negs) < self.num_neg and tries < 1000:
            # Sample random item id
            j = self.rng.randrange(self.num_items)

            # Ensure it’s not a known positive and not the same as i_pos
            if j not in pos_set and j != i_pos:
                negs.append(j)

            tries += 1

        # Fallback if user has too many positives (rare in our filtered dataset)
        if len(negs) < self.num_neg:
            while len(negs) < self.num_neg:
                negs.append(self.rng.randrange(self.num_items))

        # Return tensors (DataLoader will batch them)
        return {
            "user_id": torch.tensor(u, dtype=torch.long),
            "pos_item_id": torch.tensor(i_pos, dtype=torch.long),
            "neg_item_id": torch.tensor(negs, dtype=torch.long),  # shape: [num_neg]
        }

# -----------------------------
# Build training positives and user->pos lookup
# -----------------------------
train_pos_df = train_df[train_df["label"] == 1].copy()

# Map each user to the set of items they liked in training
user_to_train_pos = (
    train_pos_df.groupby("user_idx")["item_idx"]
    .apply(set)
    .to_dict()
)

# Create dataset and loader
num_items = int(mini["item_idx"].nunique())

train_dataset = GoodreadsTwoTowerDataset(
    df_pos=train_pos_df,
    user_to_pos_set=user_to_train_pos,
    num_items=num_items,
    num_neg=10,   # typically 5–20
    seed=42,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=1024,
    shuffle=True,
    drop_last=True,  # stable batch shapes
    num_workers=0,   # use 0 for notebook simplicity; increase for speed
)

# Sanity check a batch
batch = next(iter(train_loader))
print({k: v.shape for k, v in batch.items()})
print({k: v.dtype for k, v in batch.items()})

{'user_id': torch.Size([1024]), 'pos_item_id': torch.Size([1024]), 'neg_item_id': torch.Size([1024, 10])}
{'user_id': torch.int64, 'pos_item_id': torch.int64, 'neg_item_id': torch.int64}


In [12]:
import math
import torch.nn as nn
import torch.nn.functional as F

class TwoTowerModel(nn.Module):
    """
    Two-tower recommender with:
      - User ID embedding
      - Item ID embedding
      - Item metadata embeddings (language, year bucket, author)
      - User/item bias terms
      - Scaled dot-product scoring
    """
    def __init__(
        self,
        num_users: int,
        num_items: int,
        num_langs: int,
        num_year_buckets: int,
        num_authors: int,
        emb_dim: int = 64,
        lang_emb_dim: int = 4,
        year_emb_dim: int = 4,
        author_emb_dim: int = 8,
    ):
        super().__init__()

        # -----------------------------
        # Core ID embeddings
        # -----------------------------
        self.user_emb = nn.Embedding(num_users, emb_dim)
        self.item_emb = nn.Embedding(num_items, emb_dim)

        # -----------------------------
        # Bias terms (scalars) for popularity/activity effects
        # -----------------------------
        self.user_bias = nn.Embedding(num_users, 1)
        self.item_bias = nn.Embedding(num_items, 1)

        # -----------------------------
        # Metadata embeddings
        # -----------------------------
        self.lang_emb = nn.Embedding(num_langs, lang_emb_dim)
        self.year_emb = nn.Embedding(num_year_buckets, year_emb_dim)
        self.author_emb = nn.Embedding(num_authors, author_emb_dim)

        # -----------------------------
        # Project metadata -> emb_dim
        # -----------------------------
        # Concatenate [lang_emb, year_emb, author_emb] -> MLP -> emb_dim
        self.item_feat = nn.Sequential(
            nn.Linear(lang_emb_dim + year_emb_dim + author_emb_dim, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Linear(64, emb_dim),
        )

        # Scale factor for dot product (stabilizes logits)
        self.scale = math.sqrt(emb_dim)

        # Initialize biases to zero (common practice)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    # -----------------------------
    # Encoders
    # -----------------------------
    def encode_user(self, user_ids: torch.Tensor) -> torch.Tensor:
        """Return user embeddings: shape [B, D]."""
        return self.user_emb(user_ids)

    def encode_item(
        self,
        item_ids: torch.Tensor,
        lang_ids: torch.Tensor,
        year_ids: torch.Tensor,
        author_ids: torch.Tensor,
    ) -> torch.Tensor:
        """
        Return item embeddings enriched with metadata:
          v = v_id + f(metadata)
        """
        # Item id embedding
        v_id = self.item_emb(item_ids)  # [B, D]

        # Metadata embeddings
        v_lang = self.lang_emb(lang_ids)        # [B, L]
        v_year = self.year_emb(year_ids)        # [B, Y]
        v_author = self.author_emb(author_ids)  # [B, A]

        # Concatenate metadata vectors
        x = torch.cat([v_lang, v_year, v_author], dim=-1)  # [B, L+Y+A]

        # Project metadata into item embedding space
        v_feat = self.item_feat(x)  # [B, D]

        # Combine memorization (id) + generalization (metadata)
        return v_id + v_feat

    # -----------------------------
    # Scoring forward pass
    # -----------------------------
    def forward(
        self,
        user_ids: torch.Tensor,
        item_ids: torch.Tensor,
        lang_ids: torch.Tensor,
        year_ids: torch.Tensor,
        author_ids: torch.Tensor,
    ) -> torch.Tensor:
        """
        Compute scores for (user, item) pairs.
        Supports broadcasting shapes if user_ids/item_ids are broadcastable.
        """
        # Get user and item representations
        u = self.encode_user(user_ids)
        v = self.encode_item(item_ids, lang_ids, year_ids, author_ids)

        # Scaled dot product similarity
        dot = (u * v).sum(dim=-1) / self.scale

        # Add user/item biases
        bu = self.user_bias(user_ids).squeeze(-1)
        bi = self.item_bias(item_ids).squeeze(-1)

        return dot + bu + bi

In [13]:

def train(
    model,
    dataloader,
    optimizer,
    device,
    item_lang_ids,      # [num_items]
    item_year_ids,      # [num_items]
    item_author_ids,    # [num_items]
    info_nce=True,
    temperature=1.0,
):
    """
    Train for one epoch.

    For each batch:
      - compute s_pos for (u, pos_item)
      - compute s_neg for (u, neg_items)
      - InfoNCE: cross-entropy over [s_pos, s_neg...]
    """
    model.train()

    total_loss = 0.0
    steps = 0

    # Move item metadata tensors to device once
    item_lang_ids = item_lang_ids.to(device)
    item_year_ids = item_year_ids.to(device)
    item_author_ids = item_author_ids.to(device)

    for batch in dataloader:
        # Batch tensors
        u = batch["user_id"].to(device)           # [B]
        i_pos = batch["pos_item_id"].to(device)   # [B]
        i_neg = batch["neg_item_id"].to(device)   # [B, Nneg]

        # Lookup metadata ids for positive items
        pos_lang   = item_lang_ids[i_pos]         # [B]
        pos_year   = item_year_ids[i_pos]         # [B]
        pos_author = item_author_ids[i_pos]       # [B]

        # Lookup metadata ids for negative items
        neg_lang   = item_lang_ids[i_neg]         # [B, Nneg]
        neg_year   = item_year_ids[i_neg]         # [B, Nneg]
        neg_author = item_author_ids[i_neg]       # [B, Nneg]

        # Clear gradients
        optimizer.zero_grad(set_to_none=True)

        # Positive scores s(u, i_pos): [B]
        s_pos = model(u, i_pos, pos_lang, pos_year, pos_author)

        # Expand users to match the negative item tensor shape
        # u_neg: [B, Nneg]
        u_neg = u.unsqueeze(1).expand_as(i_neg)

        # Negative scores s(u, i_neg): [B, Nneg]
        s_neg = model(u_neg, i_neg, neg_lang, neg_year, neg_author)
   
        # InfoNCE / sampled softmax:
        # candidates scores: [B, 1 + Nneg], label is 0 (positive at index 0)
        scores = torch.cat([s_pos.unsqueeze(1), s_neg], dim=1)  # [B, 1+Nneg]
        scores = scores / temperature                           # optional sharpening
        targets = torch.zeros(scores.size(0), dtype=torch.long, device=device)
        loss = F.cross_entropy(scores, targets)

        # Backprop + optimizer step
        loss.backward()
        optimizer.step()

        total_loss += float(loss.item())
        steps += 1

    return total_loss / max(steps, 1)

In [14]:
@torch.no_grad()
def recall_at_k_mask_seen(
    model,
    eval_pos,            # dict: user -> set of held-out positive items
    train_seen,          # dict: user -> set of training-seen items to mask
    num_items,
    item_lang_ids,       # torch tensor of shape [num_items]
    item_year_ids,       # torch tensor of shape [num_items]
    item_author_ids,     # torch tensor of shape [num_items]
    K=20,
    device="cpu",
    max_users=None,
):
    """
    Compute average per-user Recall@K with masking of items seen in training.

    Steps per user:
      - Score all items
      - Mask train_seen items (set their scores to -inf)
      - Take top-K
      - Recall = hits_in_topK / num_heldout_positives
    """
    model.eval()

    # Move item metadata tensors to device once
    item_lang_ids = item_lang_ids.to(device)
    item_year_ids = item_year_ids.to(device)
    item_author_ids = item_author_ids.to(device)

    # All item ids [0..num_items-1]
    all_item_ids = torch.arange(num_items, device=device, dtype=torch.long)

    # Precompute all item embeddings and biases (fast for repeated user scoring)
    all_item_embs = model.encode_item(
        all_item_ids,
        item_lang_ids,
        item_year_ids,
        item_author_ids
    )  # [I, D]
    all_item_bias = model.item_bias(all_item_ids).squeeze(-1)  # [I]

    # User list to evaluate
    users = list(eval_pos.keys())
    if max_users is not None:
        users = users[:max_users]

    recalls = []

    for u in users:
        # Held-out positives for this user
        pos_set = eval_pos.get(u, set())
        if not pos_set:
            continue

        # Compute user embedding + bias
        user_id = torch.tensor([u], device=device, dtype=torch.long)
        user_emb = model.encode_user(user_id)                # [1, D]
        user_b   = model.user_bias(user_id).squeeze(-1)      # [1]

        # Dot-product scores for all items: [I]
        scores = (user_emb @ all_item_embs.T).squeeze(0) / model.scale

        # Add item biases and user bias
        scores = scores + all_item_bias + user_b

        # Mask training-seen items so we don’t “recommend” what they already consumed
        seen = train_seen.get(u, set())
        if seen:
            seen_idx = torch.tensor(list(seen), device=device, dtype=torch.long)
            scores[seen_idx] = -1e9

        # Top-K predicted item ids
        topk = torch.topk(scores, K, largest=True).indices.tolist()

        # Count hits among held-out positives
        hits = sum((i in pos_set) for i in topk)
        recalls.append(hits / len(pos_set))

    # Average recall across users
    return float(sum(recalls) / len(recalls)) if recalls else 0.0

In [15]:
# -----------------------------
# Choose device: MPS (Mac), CUDA, or CPU
# -----------------------------
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Using device:", device)

# -----------------------------
# Build item-level metadata tensors aligned to item_idx
# -----------------------------
# We need a single row per item_idx containing its lang/year/author indices.
item_meta = (
    mini[["item_idx", "lang_idx", "year_idx", "author_idx"]]
    .drop_duplicates("item_idx")
    .sort_values("item_idx")
)

# Convert metadata columns into torch tensors of shape [num_items]
item_lang_ids   = torch.tensor(item_meta["lang_idx"].values, dtype=torch.long)
item_year_ids   = torch.tensor(item_meta["year_idx"].values, dtype=torch.long)
item_author_ids = torch.tensor(item_meta["author_idx"].values, dtype=torch.long)

# Cardinalities
num_users = int(mini["user_idx"].nunique())
num_items = int(mini["item_idx"].nunique())
num_langs = int(mini["lang_idx"].nunique())
num_year_buckets = int(mini["year_idx"].nunique())
num_authors = int(mini["author_idx"].nunique())

# Sanity checks: item_meta should be exactly one row per item, and ordered 0..num_items-1
assert len(item_meta) == num_items
assert (item_meta["item_idx"].values == torch.arange(num_items).numpy()).all()

# -----------------------------
# Build train_seen (mask set) and eval_pos (held-out positives)
# -----------------------------
train_seen = (
    train_df.groupby("user_idx")["item_idx"]
    .apply(set)
    .to_dict()
)

# Evaluate on validation set during training
eval_df = val_df

# Held-out positives per user in eval split
eval_pos = (
    eval_df[eval_df["label"] == 1]
    .groupby("user_idx")["item_idx"]
    .apply(set)
    .to_dict()
)

print("train users:", len(train_seen), "eval users:", len(eval_pos))

# -----------------------------
# Initialize model + optimizer
# -----------------------------
model = TwoTowerModel(
    num_users=num_users,
    num_items=num_items,
    num_langs=num_langs,
    num_year_buckets=num_year_buckets,
    num_authors=num_authors,
    emb_dim=64,
).to(device)

# Adam with mild weight decay (L2 regularization)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

# -----------------------------
# Train loop
# -----------------------------
num_epochs = 5
losses = []
recall20s = []

for epoch in range(1, num_epochs + 1):
    # Train one epoch
    loss = train(
        model=model,
        dataloader=train_loader,
        optimizer=optimizer,
        device=device,
        item_lang_ids=item_lang_ids,
        item_year_ids=item_year_ids,
        item_author_ids=item_author_ids,
        info_nce=True,         # sampled softmax
        temperature=1.0,       # can tune
    )

    # Evaluate masked recall@20
    r20 = recall_at_k_mask_seen(
        model=model,
        eval_pos=eval_pos,
        train_seen=train_seen,
        num_items=num_items,
        item_lang_ids=item_lang_ids,
        item_year_ids=item_year_ids,
        item_author_ids=item_author_ids,
        K=20,
        device=device,
        max_users=2000,        # cap for speed; set None for full eval
    )

    losses.append(loss)
    recall20s.append(r20)

    print(f"Epoch {epoch:02d} | loss: {loss:.4f} | masked recall@20: {r20:.4f}")

# -----------------------------
# Plot loss vs masked recall@20
# -----------------------------
epochs = list(range(1, num_epochs + 1))

fig, ax1 = plt.subplots(figsize=(15, 5))

# Left axis: loss
ax1.plot(epochs, losses, marker="o", label="Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.grid(True)

# Right axis: recall@20
ax2 = ax1.twinx()
ax2.plot(epochs, recall20s, marker="o", label="Masked Recall@20")
ax2.set_ylabel("Masked Recall@20")

# Combine legends from both axes
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="best")

plt.title("Training Progress: Loss vs Masked Recall@20")
plt.tight_layout()
plt.show()

Using device: cuda
train users: 24355 eval users: 18046
Epoch 01 | loss: 2.6478 | masked recall@20: 0.0021
Epoch 02 | loss: 2.4603 | masked recall@20: 0.0015
Epoch 03 | loss: 2.4034 | masked recall@20: 0.0014
Epoch 04 | loss: 2.3870 | masked recall@20: 0.0012
Epoch 05 | loss: 2.3818 | masked recall@20: 0.0014


NameError: name 'plt' is not defined

In [ ]:
import numpy as np
import torch

def recall_at_k_pop_mask_seen(eval_pos, train_seen, popular_items, K=20, max_users=None):
    users = list(eval_pos.keys())
    if max_users is not None:
        users = users[:max_users]

    recalls = []
    for u in users:
        pos_set = eval_pos.get(u, set())
        if not pos_set:
            continue

        seen = train_seen.get(u, set())

        recs = []
        for it in popular_items:
            if it not in seen:
                recs.append(it)
            if len(recs) >= K:
                break

        hits = sum((i in pos_set) for i in recs)
        recalls.append(hits / len(pos_set))

    return float(sum(recalls) / len(recalls)) if recalls else 0.0

# -----------------------------
# 1) Build train_seen and eval_pos
# -----------------------------
train_seen = (
    train_df.groupby("user_idx")["item_idx"]
    .apply(set)
    .to_dict()
)

# choose split to evaluate on
eval_df = test_df   # <-- change to val_df if you want validation

eval_pos = (
    eval_df[eval_df["label"] == 1]
    .groupby("user_idx")["item_idx"]
    .apply(set)
    .to_dict()
)


# -----------------------------
# 2) Popularity list from TRAIN positives (global)
# -----------------------------
popular_items = (
    train_df[train_df["label"] == 1]
    .groupby("item_idx")
    .size()
    .sort_values(ascending=False)
    .index.tolist()
)


# -----------------------------
# 4) Compute both metrics on SAME users (cap optional)
# -----------------------------
MAX_USERS = 2000  # set None for all users (slower but best)

model_r20 = recall_at_k_mask_seen(
    model=model,
    eval_pos=eval_pos,
    train_seen=train_seen,
    num_items=num_items,
    item_lang_ids=item_lang_ids,
    item_year_ids=item_year_ids,
    item_author_ids=item_author_ids,
    K=20,
    device=device,
    max_users=MAX_USERS,
)

pop_r20 = recall_at_k_pop_mask_seen(
    eval_pos=eval_pos,
    train_seen=train_seen,
    popular_items=popular_items,
    K=20,
    max_users=MAX_USERS,
)

print(f"Masked Recall@20 (Model): {model_r20:.4f}")
print(f"Masked Recall@20 (Pop)  : {pop_r20:.4f}")
print(f"Lift (abs): {model_r20 - pop_r20:+.4f}")
print(f"Lift (rel): {(model_r20 / pop_r20 - 1.0)*100:+.2f}%")